# Blog 1: One Small Model, One Function-Calling Task Slice

This is the exploratory notebook for Blog 1. It is intentionally **not** a full benchmark runner. The long-running steps live in Python scripts; this notebook is for seeing one or two examples clearly.

We use a deterministic NESTFUL slice:

- dataset: `ibm-research/nestful`
- local split: deterministic `80/20`, seed `42`
- filter: reference solution has `<= 2` function calls
- train rows: `506`
- eval rows: `103`

The goal is to understand the loop: model input -> model output -> parser -> normalized calls -> deterministic function execution -> score.

In [ ]:
from pathlib import Path
import json
import sys

start = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate
    for candidate in [start, *start.parents]
    if (candidate / "common" / "nestful.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common import config as cfg
from common import nestful

DATA_DIR = PROJECT_ROOT / "data" / "nestful_calls_le_2"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

if not DATA_DIR.exists():
    nestful.prepare_data(DATA_DIR, max_calls=2)

train_rows = nestful.load_prepared_rows(DATA_DIR, "train")
eval_rows = nestful.load_prepared_rows(DATA_DIR, "eval")
stats = json.loads((DATA_DIR / "stats.json").read_text())

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Train rows:", len(train_rows))
print("Eval rows:", len(eval_rows))
print(json.dumps(stats, indent=2)[:2000])

## What Kind Of Distillation Is This?

This first blog uses **offline hard-token distillation**. Offline means the training data is generated before student training starts. Hard-token means the student is trained on the teacher's final text tokens, not on the teacher's probability distribution or logits.

For this benchmark, the target text is a JSON array of function calls. If a strong teacher emits the correct call sequence, we keep that row as supervised fine-tuning data. Later posts can add softer signals such as probabilities/logits or on-policy data, but this first pass should stay simple enough that we can inspect the whole mechanism.

## The Harness Boundary

NESTFUL is not a multi-turn chat environment. The model emits the whole nested function-call plan in **one assistant turn**. Then the harness parses and executes the calls in order.

```mermaid
flowchart LR
    A["Dataset row
user query + tool catalog"] --> B["Chat template
messages to model"]
    B --> C["Model generation
JSON call array"]
    C --> D["Parser
valid JSON? normalize labels?"]
    D --> E["Function executor
resolve $var_1.output$ references"]
    E --> F["Score
exact sequence + name sequence"]
```

So the notebook below shows the full cycle for one row, but the environment does not talk back to the model after each tool call. The nested references inside the generated JSON are how the second call consumes the first call's result.

![Sketchnote harness boundaries](../assets/harness-boundaries.png)

## Parser v2: the harness bug we had to fix

The first scorer accepted only one strict JSON array. That made the base student look worse than it really was, because many of its outputs were sequences of top-level JSON objects. That is still structured JSON; it is just not wrapped in `[...]`.

The fixed parser accepts these structured forms:

- one JSON array: `[{...}, {...}]`
- one object with calls: `{"calls": [{...}]}`
- one call object: `{...}`
- multiple top-level JSON call objects, decoded in sequence

It still does not repair invalid JSON, guess missing quotes, or use wording-specific cleanup. This is the fair harness used for the final comparisons.

![Sketchnote RL boundary view](../assets/rl-environment-loop.png)

Later in the series, this same boundary becomes the RL loop: the model emits an action, the harness parses it, the simulator executes it, and the reward/scorer gives the trainer a learning signal. In Blog 1 we stay simpler: deterministic scoring plus offline SFT rows.

## Pick One Case

We choose one eval row that has a saved successful GPT teacher output, because it lets us inspect a real model output that flows all the way through parsing and execution. If the saved reports are missing, the notebook still shows the reference solution.

In [ ]:
report_paths = {
    "student": OUTPUT_DIR / "qwen3_5_0_8b_mlx_nestful_calls_le_2_eval_parser_v2.json",
    "qwen_teacher": OUTPUT_DIR / "qwen3_5_35b_a3b_8bit_mlx_server_nestful_calls_le_2_eval_parser_v2.json",
    "gpt_teacher": OUTPUT_DIR / "gpt_5_5_medium_nestful_calls_le_2_eval_parser_v2.json",
    "lfm_teacher": OUTPUT_DIR / "lfm2_5_8b_a1b_mlx_nestful_calls_le_2_eval_parser_v2.json",
    "sft_student": OUTPUT_DIR / "qwen3_5_0_8b_mlx_nestful_calls_le_2_qwen_teacher_sft_parser_v2_b1_eval.json",
}
reports = {}
for name, path in report_paths.items():
    if path.exists():
        reports[name] = json.loads(path.read_text())
        print(name, "summary:", json.dumps(reports[name]["summary"], indent=2))
    else:
        print(name, "report missing:", path)

case_id = eval_rows[0]["id"]
if "gpt_teacher" in reports:
    exact_gpt_rows = [item for item in reports["gpt_teacher"]["results"] if item.get("exact_correct")]
    if exact_gpt_rows:
        case_id = exact_gpt_rows[0]["id"]

row = next(item for item in eval_rows if item["id"] == case_id)
student_result = None
gpt_result = None
if "student" in reports:
    student_result = next((item for item in reports["student"]["results"] if item["id"] == case_id), None)
if "gpt_teacher" in reports:
    gpt_result = next((item for item in reports["gpt_teacher"]["results"] if item["id"] == case_id), None)

print("\nSelected case:", case_id)
print("User query:")
print(row["input"])
print("\nExpected reference calls:")
print(json.dumps(row["expected_calls"], indent=2, ensure_ascii=False))

## What The Model Receives

Both the student and teacher receive the same task messages: one system message containing the function-calling contract and the full available tool catalog for this row, then one user message with the problem.

In [ ]:
model_input_messages = row["messages"][:-1]
print(json.dumps(model_input_messages, indent=2, ensure_ascii=False))
print("\nMessage count:", len(model_input_messages))
print("Character count:", sum(len(message["content"]) for message in model_input_messages))

## Student Cycle From Saved Output

This is the small base model's saved output for the selected row. We do not execute anything if parsing fails; that is the correct harness behavior. A malformed call array cannot be safely passed to the environment.

In [ ]:
if student_result is None:
    print("No saved student result for", case_id)
else:
    raw_student_output = student_result["raw_output"]
    parsed_student_calls = nestful.parse_call_sequence(raw_student_output)
    student_score = nestful.score_output(row, raw_student_output)

    print("Raw student output:")
    print(raw_student_output)
    print("\nParsed / cleaned calls:")
    print(json.dumps(parsed_student_calls, indent=2, ensure_ascii=False))
    print("\nScore:")
    print(json.dumps({key: student_score[key] for key in ["error", "name_sequence_correct", "exact_correct"]}, indent=2))

    if parsed_student_calls is None:
        print("\nEnvironment step skipped: parser returned None.")
    else:
        student_env_trace = nestful.execute_call_sequence(row, parsed_student_calls)
        print("\nEnvironment execution trace:")
        print(json.dumps(student_env_trace, indent=2, ensure_ascii=False))

## Teacher Cycle From Saved Output

Now compare the same row with the saved GPT teacher output. Here the parser succeeds, the normalized calls match the reference, and the environment can execute the nested call sequence.

In [ ]:
if gpt_result is None:
    print("No saved GPT teacher result for", case_id)
else:
    raw_teacher_output = gpt_result["raw_output"]
    parsed_teacher_calls = nestful.parse_call_sequence(raw_teacher_output)
    teacher_score = nestful.score_output(row, raw_teacher_output)

    print("Raw teacher output:")
    print(raw_teacher_output)
    print("\nParsed / cleaned calls:")
    print(json.dumps(parsed_teacher_calls, indent=2, ensure_ascii=False))
    print("\nScore:")
    print(json.dumps({key: teacher_score[key] for key in ["error", "name_sequence_correct", "exact_correct"]}, indent=2))

    if parsed_teacher_calls is None:
        print("\nEnvironment step skipped: parser returned None.")
    else:
        teacher_env_trace = nestful.execute_call_sequence(row, parsed_teacher_calls)
        print("\nEnvironment execution trace:")
        print(json.dumps(teacher_env_trace, indent=2, ensure_ascii=False))

## Reference Execution

The reference answer is what the benchmark expects. Executing it makes the nested variable references concrete: `$var_2` can use the output produced by `$var_1`.

In [ ]:
reference_env_trace = nestful.execute_call_sequence(row, row["expected_calls"])
print(json.dumps(reference_env_trace, indent=2, ensure_ascii=False))

## Optional Live One-Row Student Probe

The cells above use saved outputs, so the notebook stays fast. If you want to run the local MLX student live for this same single row, flip the flag below. This still runs only one case.

In [ ]:
RUN_LIVE_STUDENT = False

if RUN_LIVE_STUDENT:
    from common import generation

    generate_student = generation.make_generator(
        backend="mlx",
        model_name=cfg.MLX_STUDENT_MODEL,
        max_new_tokens=512,
    )
    live_student_output = generate_student(row)
    live_student_calls = nestful.parse_call_sequence(live_student_output)
    live_student_score = nestful.score_output(row, live_student_output)

    print("Live raw student output:")
    print(live_student_output)
    print("\nParsed / cleaned calls:")
    print(json.dumps(live_student_calls, indent=2, ensure_ascii=False))
    print("\nScore:")
    print(json.dumps({key: live_student_score[key] for key in ["error", "name_sequence_correct", "exact_correct"]}, indent=2))

    if live_student_calls is not None:
        print("\nEnvironment execution trace:")
        print(json.dumps(nestful.execute_call_sequence(row, live_student_calls), indent=2, ensure_ascii=False))
    else:
        print("\nEnvironment step skipped: parser returned None.")
else:
    print("RUN_LIVE_STUDENT is False, so this notebook is only inspecting saved one-row outputs.")

## Final Results

![Sketchnote result ladder](../assets/blog1-results-sketchnote.png)

All rows below use the same `data/nestful_calls_le_2/eval.jsonl` split and the same parser v2 harness.

| Run | Exact | Name Sequence | Parse Rate |
| --- | ---: | ---: | ---: |
| Base Qwen3.5-0.8B MLX | 2/103 | 7/103 | 83/103 |
| Qwen3.5-35B-A3B 8-bit teacher | 37/103 | 39/103 | 98/103 |
| GPT 5.5 medium teacher | 69/103 | 78/103 | 103/103 |
| LFM2.5-8B-A1B MLX 8-bit baseline | 0/103 | 0/103 | 0/103 |
| Qwen3.5-0.8B MLX after Qwen-teacher SFT | 43/103 | 53/103 | 99/103 |

The fine-tuned student beating the zero-shot Qwen teacher is not a claim about general intelligence. It means the student specialized to this exact-format, short-call NESTFUL slice after training on filtered correct teacher rows.

## How Teacher Rows Become SFT Data

![Sketchnote teacher filtered SFT rows](../assets/teacher-filtered-sft.png)

The Qwen teacher ran on the train split. We kept only exact train examples: `156/506`. Those rows became supervised examples where the prompt is the same user query + tool catalog, and the target is the teacher's exact JSON call sequence.

## Saved Baseline Results

These are the full eval reports already produced by scripts. They are here only as context; this notebook does not recompute them.

In [ ]:
for name, report in reports.items():
    print("\n" + name)
    print(json.dumps(report["summary"], indent=2))

## Before/After SFT Example

This cell looks for one held-out eval row where the base student failed and the SFT student was exact. It is useful for seeing the behavior change without running a new model.

In [ ]:
base_results = {item["id"]: item for item in reports.get("student", {}).get("results", [])}
sft_results = {item["id"]: item for item in reports.get("sft_student", {}).get("results", [])}
improved_ids = [
    row_id
    for row_id, sft_item in sft_results.items()
    if sft_item.get("exact_correct") and not base_results.get(row_id, {}).get("exact_correct")
]

if not improved_ids:
    print("No improved examples found in the loaded reports.")
else:
    improved_id = improved_ids[0]
    improved_row = next(item for item in eval_rows if item["id"] == improved_id)
    print("Improved held-out row:", improved_id)
    print("User query:")
    print(improved_row["input"])
    print("\nExpected:")
    print(json.dumps(improved_row["expected_calls"], indent=2, ensure_ascii=False))
    print("\nBase student raw output:")
    print(base_results[improved_id]["raw_output"])
    print("\nSFT student raw output:")
    print(sft_results[improved_id]["raw_output"])
    print("\nSFT parsed calls:")
    print(json.dumps(sft_results[improved_id]["predicted"], indent=2, ensure_ascii=False))

## What Comes Next

After this inspection notebook, use scripts for the real experiment:

1. Run the student baseline on the held-out eval split.
2. Run teacher baselines.
3. Generate teacher SFT rows from successful train split outputs.
4. Train the student.
5. Evaluate the trained student on the same held-out eval split.

The important lesson from this notebook is the contract. A student can only improve if we train it to emit the exact JSON call sequence the harness can parse and execute.